pandas: Python's Data Analysis Toolkit
--------------------------------------

So far, we have explored foundational concepts common to all programming languages and workflows. Useful stuff, but up to this point we have been working with each concept in isolation. This helps to get the hang of things, but now we are going to use these tools to build a real-life workflow.

For the rest of today's workshop, we are going to develop a high level analysis of the [NYC Squirrel Census](https://www.thesquirrelcensus.com/) dataset. The data are hosted at the [New York City Open Data](https://data.cityofnewyork.us/d/vfnx-vebw) catalog. Additional data about hectares and stories can be found using the [results from this query string](https://data.cityofnewyork.us/browse?sortBy=relevance&pageSize=20&q=squirrel+census&page=1).

The workflow we'll demonstrate is typical of data science workflows and includes:

1. Accessing remote or networked data and saving to a local file for future reference.
2. Inspecting data to assess completeness, key attributes, and fitness for purpose.
3. Generating summary statistics and plots to understand the distribution(s) of different variables.
4. Grouping data by different categories and generating aggregate statistics.
5. Visualizing data.
6. Exploring correlations between variables (as time allows!)

Will we be able to fit models for further analysis? Probably not today! But there are some powerful Python tools and libraries for modeling and machine learning. We will post information about these resources in the Google Doc.

## Citation
-----------

_Cite squirrel census data here_

## Accessing and saving data
----------------------------

In [1]:
# import needed libraries
import pandas as pd
import requests
import io

In [2]:
# Go to the data - we are not downloading the data from the website! Well, we are but we are using an API
# https://data.cityofnewyork.us/browse?sortBy=relevance&pageSize=20&q=squirrel+census&page=1
# walk through the 3 datasets - data, stories, hectare data - including variable definitions
# demonstrate API - version 2 for us, no authentication, 1K row limit on census data needs to be worked around

# do stories first - smallest of the datasets
# API endpoint, JSON, Soda 2, copy to clipboard
stories_url = "https://data.cityofnewyork.us/resource/gfqj-f768.json"

In [3]:
# API has a 1K row limit but there are 809 rows in this dataset so we should be able to get it all with one API call
r = requests.get(stories_url)

In [4]:
# What is this?

In [5]:
print(type(r))

<class 'requests.models.Response'>


In [40]:
# What can we do with this? How do we get the data included in the response?
# help(r)

In [7]:
# inspect - boolean value to confirm success of request (a 2** response)
print(r.ok)

True


In [9]:
# actual status_code - 200 is good! Did anyone not get a 200?
print(r.status_code)

200


In [41]:
# what else can we look at?
# print(r.content)

In [42]:
# print(r.text)

In [43]:
# we used the JSON API endpoint - requests has a method for inspecting the JSON
# print(r.json())

In [13]:
# all looks good - convert to a pandas dataframe
squirrels_stories = pd.DataFrame(r.json())

In [15]:
# now look - the json has been serilialized as a tabular data structure (a dataframe)
squirrels_stories

,hectare,shift,date,note_squirrel_park_stories,story_topic_squirrel,story_topic_park_experience,story_topic_other_animals,story_topic_other,story_topic_census_takers,story_topic_squirrels_acting,story_topic_dogs,story_topic_accidental_poems
0,01A,PM,10142018,Observed a squirrel with a cache of peanuts th...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01A,AM,10182018,"Lots of pedestrians and vehicular traffic, foo...",NaN,True,NaN,NaN,NaN,NaN,NaN,NaN
2,01A,PM,10202018,Three of the pedicab drivers (14 pedicabs line...,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN
3,01A,AM,10072018,Squirrels were in surrounding grass areas/tree...,True,True,True,Birds,NaN,NaN,NaN,NaN
4,01B,AM,10122018,"""SQUIRREL APALOOZA""\n\n6+7 were playing on tre...",True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
804,42H,PM,10142018,"grey body, red head — exquisite. These two app...",True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
805,42H,AM,10192018,Very quiet for squirrel activity but glad I ma...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
806,42H,AM,10142018,Homeless couple under tree\nBunch of mallards ...,NaN,True,NaN,NaN,NaN,NaN,NaN,True
807,42I,PM,10142018,I saw squirrel #1 chase squirrel #2 away from ...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# sanity check - make sure we have 809 rows
# refer to RangeIndex - not all rows have values for all cols (see for example the "topic" columns)
print(squirrels_stories.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 809 entries, 0 to 808
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   hectare                       809 non-null    object
 1   shift                         809 non-null    object
 2   date                          809 non-null    object
 3   note_squirrel_park_stories    809 non-null    object
 4   story_topic_squirrel          407 non-null    object
 5   story_topic_park_experience   468 non-null    object
 6   story_topic_other_animals     134 non-null    object
 7   story_topic_other             94 non-null     object
 8   story_topic_census_takers     31 non-null     object
 9   story_topic_squirrels_acting  24 non-null     object
 10  story_topic_dogs              164 non-null    object
 11  story_topic_accidental_poems  64 non-null     object
dtypes: object(12)
memory usage: 76.0+ KB
None


In [17]:
# we have data - let's save locally so we don't have to download every time we work with the data
# there are reasons for saving locally or not - depends on what you are doing
squirrels_stories.to_csv('./2018_nyc_squirrels_stories.csv', index=False)

In [18]:
# okay now get the hectare data, which can also be done with one API call
hectare_url = "https://data.cityofnewyork.us/resource/ej9h-v6g2.json"

In [19]:
# same process as before
# in fact - we could write a function but for today we will do it manually
r = requests.get(hectare_url)

In [20]:
squirrels_hectares = pd.DataFrame(r.json())

In [21]:
squirrels_hectares

,hectare,shift,date,anonymized_sighter,sighter_observed_weather,litter,other_animal_sightings,hectare_conditions,number_of_sighters,number_of_squirrels,total_time_of_sighting,hectare_conditions_notes,litter_notes
0,01A,AM,10072018,110,"70º F, Foggy",Some,"Humans, Pigeons",Busy,1,4,22,NaN,NaN
1,01A,PM,10142018,177,"54º F, overcast",Abundant,"Humans, Pigeons",Busy,1,7,26,NaN,NaN
2,01B,AM,10122018,11,"60º F, sunny",Some,"Humans, Dogs, Pigeons, Horses",Busy,1,17,23,NaN,NaN
3,01B,PM,10192018,109,"59.8º F, Sun, Cool",Some,"Humans, Dogs, Pigeons, Sparrow, Blue jay",Busy,1,10,35,NaN,NaN
4,01C,PM,10132018,241,"55° F, Partly Cloudy",None,"Humans, Dogs, Pigeons, Birds",Busy,1,10,25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,09A,AM,10132018,179180181,NaN,Some,"Humans, Dogs, Pigeons, Rats, Blue jays",Busy,3,12,20,NaN,NaN
696,11F,PM,10062018,168,NaN,Some,"Humans, Dogs, Pigeons",Busy,3,3,26,NaN,NaN
697,12G,PM,10072018,106.107108,80º F,None,"Humans, Cat bird",Busy,3,7,29,NaN,NaN
698,12H,PM,10072018,106.107108,79º F,None,"Humans, Sparrows",Busy,3,8,30,NaN,NaN


In [23]:
print(squirrels_hectares.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   hectare                   700 non-null    object
 1   shift                     700 non-null    object
 2   date                      700 non-null    object
 3   anonymized_sighter        700 non-null    object
 4   sighter_observed_weather  681 non-null    object
 5   litter                    610 non-null    object
 6   other_animal_sightings    668 non-null    object
 7   hectare_conditions        660 non-null    object
 8   number_of_sighters        700 non-null    object
 9   number_of_squirrels       700 non-null    object
 10  total_time_of_sighting    680 non-null    object
 11  hectare_conditions_notes  74 non-null     object
 12  litter_notes              6 non-null      object
dtypes: object(13)
memory usage: 71.2+ KB
None


In [24]:
squirrels_hectares.to_csv("./2018_nyc_squirrels_hectares.csv", index=False)

In [25]:
# and now for the actual census data - obs of individual squirrels
# 3023 rows, so we have to work around the API row limit - the documentation tells us how to do this (review documentation)
# rows 1-1000
squirrels_url_0 = "https://data.cityofnewyork.us/resource/vfnx-vebw.json"

# rows 1000-1999
squirrels_url_1 = "https://data.cityofnewyork.us/resource/vfnx-vebw.json?$offset=1000"

# rows 2000-2999
squirrels_url_2 = "https://data.cityofnewyork.us/resource/vfnx-vebw.json?$offset=2000"

# rows 3000-3023
squirrels_url_3 = "https://data.cityofnewyork.us/resource/vfnx-vebw.json?$offset=3000"

In [26]:
# there are multiple ways we could do this - here's one that uses the concepts we have discussed today
url_list = [squirrels_url_0, squirrels_url_1, squirrels_url_2, squirrels_url_3]
result_list = []

for url in url_list:
    print("Getting:", url)
    r = requests.get(url)
    result_list.append(r.json())

Getting: https://data.cityofnewyork.us/resource/vfnx-vebw.json
Getting: https://data.cityofnewyork.us/resource/vfnx-vebw.json?$offset=1000
Getting: https://data.cityofnewyork.us/resource/vfnx-vebw.json?$offset=2000
Getting: https://data.cityofnewyork.us/resource/vfnx-vebw.json?$offset=3000


In [44]:
# now the result list is big and not really human readable
# print(result_list)

In [30]:
# but it converts into a dataframe just as efficiently as our smaller datasets
# what is happening here - 'concat' and argument as list of dataframes
squirrels_census = pd.concat([pd.DataFrame(result_list[0]), pd.DataFrame(result_list[1]),
                             pd.DataFrame(result_list[2]), pd.DataFrame(result_list[3])])

In [31]:
# we could have done it a more verbose way - this is equivalent
s_census_0 = pd.DataFrame(result_list[0])
s_census_1 = pd.DataFrame(result_list[1])
s_census_2 = pd.DataFrame(result_list[2])
s_census_3 = pd.DataFrame(result_list[3])
squirrels_census_alt = pd.concat([s_census_0, s_census_1, s_census_2, s_census_3])

In [34]:
# compare - empty result confirms the two dataframe are the same
# note - this isn't something you would normally do but we're just showing two ways to do something that
# produce equivalent results; this kind of thing is common in programming!
df_compare = squirrels_census.compare(squirrels_census_alt)
print(df_compare)

Empty DataFrame
Columns: []
Index: []


In [35]:
# inspect
squirrels_census

,x,y,unique_squirrel_id,hectare,shift,date,hectare_squirrel_number,combination_of_primary_and,running,chasing,...,primary_fur_color,location,above_ground_sighter,age,color_notes,highlight_fur_color,specific_location,other_activities,other_interactions,:@computed_region_efsh_h5xi
0,-73.9561344937861,40.7940823884086,37F-PM-1014-03,37F,PM,10142018,3,+,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-73.9688574691102,40.7837825208444,21B-AM-1019-04,21B,AM,10192018,4,+,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-73.9742811484852,40.775533619083,11B-PM-1014-08,11B,PM,10142018,8,Gray+,False,True,...,Gray,Above Ground,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-73.9596413903948,40.7903128889029,32E-PM-1017-14,32E,PM,10172018,14,Gray+,False,False,...,Gray,NaN,NaN,Adult,Nothing selected as Primary. Gray selected as ...,NaN,NaN,NaN,NaN,NaN
4,-73.9702676472613,40.7762126854894,13E-AM-1017-05,13E,AM,10172018,5,Gray+Cinnamon,False,False,...,Gray,Above Ground,NaN,Adult,NaN,Cinnamon,on tree stump,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,-73.9639431360458,40.7908677445466,30B-AM-1007-04,30B,AM,10072018,4,Gray+,False,False,...,Gray,Ground Plane,FALSE,Adult,NaN,NaN,NaN,NaN,NaN,NaN
19,-73.9704015859639,40.7825600069973,19A-PM-1013-05,19A,PM,10132018,5,Gray+White,False,False,...,Gray,Ground Plane,FALSE,Adult,NaN,White,NaN,NaN,NaN,NaN
20,-73.9665871993517,40.7836775064883,22D-PM-1012-07,22D,PM,10122018,7,"Gray+Black, Cinnamon, White",False,False,...,Gray,Ground Plane,FALSE,Adult,NaN,"Black, Cinnamon, White",NaN,NaN,NaN,NaN
21,-73.9639941227864,40.7899152327912,29B-PM-1010-02,29B,PM,10102018,2,"Gray+Cinnamon, White",False,False,...,Gray,Ground Plane,FALSE,NaN,NaN,"Cinnamon, White",NaN,NaN,NaN,NaN


In [36]:
print(squirrels_census.info())

<class 'pandas.core.frame.DataFrame'>
Index: 3023 entries, 0 to 22
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   x                            3023 non-null   object
 1   y                            3023 non-null   object
 2   unique_squirrel_id           3023 non-null   object
 3   hectare                      3023 non-null   object
 4   shift                        3023 non-null   object
 5   date                         3023 non-null   object
 6   hectare_squirrel_number      3023 non-null   object
 7   combination_of_primary_and   3023 non-null   object
 8   running                      3023 non-null   bool  
 9   chasing                      3023 non-null   bool  
 10  climbing                     3023 non-null   bool  
 11  eating                       3023 non-null   bool  
 12  foraging                     3023 non-null   bool  
 13  kuks                         3023 non-nu

In [37]:
# now these 'computed region' columns are just taking up memory, so drop them
# maybe don't do during workshop - that's a lot of typing
squirrels_census.drop(columns=[":@computed_region_f5dn_yrer", ":@computed_region_yeji_bk3q",
                               ":@computed_region_92fq_4b7q", ":@computed_region_sbqj_enih",
                               ":@computed_region_efsh_h5xi"], inplace=True)

In [38]:
print(squirrels_census.info())

<class 'pandas.core.frame.DataFrame'>
Index: 3023 entries, 0 to 22
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   x                           3023 non-null   object
 1   y                           3023 non-null   object
 2   unique_squirrel_id          3023 non-null   object
 3   hectare                     3023 non-null   object
 4   shift                       3023 non-null   object
 5   date                        3023 non-null   object
 6   hectare_squirrel_number     3023 non-null   object
 7   combination_of_primary_and  3023 non-null   object
 8   running                     3023 non-null   bool  
 9   chasing                     3023 non-null   bool  
 10  climbing                    3023 non-null   bool  
 11  eating                      3023 non-null   bool  
 12  foraging                    3023 non-null   bool  
 13  kuks                        3023 non-null   bool  
 14 

In [39]:
# save to file
# explain index=False
squirrels_census.to_csv('./2018_nyc_squirrels_census.csv', index=False)

This may seem like a roundabout way to download and save data, when we can often just download a CSV file directly from a website. And sometimes it makes more sense to use the manual approach - I do it all the time! 

But consider:

1. In many cases direct download of spreadsheet files is not possible. Think of the US Census! Sometimes we have to use APIs.
2. APIs may change less frequently than URLs. An API call may still work after a CSV or downloadable file has been moved.
3. Manual downloading requires you to visit the website every time you want to retrieve a file. Encoding the API parameters into a script means you can retrieve the data dynamically without having to leave your programming environment. You also don't need to locate a file that you downloaded, copy or move it to your project directory, etc. 
4. Think about reproducibility - APIs and API calls are easier to document. Consider the steps involved in downloading a file from a website and how that process would be described or documented (and consider that the process may change if the website changes!).
5. Also regarding reproducibility, scripts with API calls can be shared without also having to share the CSV files. If we download data manually, we have to include those data in any file sharing service, version control service, etc. that we may use to share data and code with collaborators. That consumes storage and network bandwidth, wheras API calls are just text in a script and use almost no additional storage or network bandwidth.
